# Python. Семинар 5

Подготовка к семинару «Функции и классы вблизи»: короткий recap лекции 5,
углубление с фишками и викторина «предскажи, что выведет». Разборы вопросов
спрятаны под спойлер — сначала ответь сам, потом раскрой.


---

## Часть 1. Быстрый recap лекции

### 1.1. Позиционные, именованные, дефолты

У параметра два свойства: как его разрешено передавать (по позиции, по имени) и
есть ли у него значение по умолчанию. Два разделителя расставляют границы: всё
до `/` — только по позиции, всё после `*` — только по имени:


In [ ]:
def linear_function(
    x: float,
    /,                    # x — только позиционный
    slope: float = 1.0,   # slope и intercept — как угодно, с дефолтами
    intercept: float = 0.0,
) -> float:
    return slope * x + intercept


print(linear_function(2))
print(linear_function(2, 10))
print(linear_function(2, intercept=0.5, slope=3))


### 1.2. Только по позиции и только по имени

`quadratic_equation(x, a, b, /, *, c=0.0)` из лекции смешивает всё сразу. Чистые
крайние случаи: имена шести координат — не часть интерфейса, их прячут за `/`;
наоборот, `settings`-подобные параметры заставляют писать имя — вызов читается
сам:


In [ ]:
import math


def euclidean_distance_3d(
    x1: float, y1: float, z1: float,
    x2: float, y2: float, z2: float, /,
) -> float:
    return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2 + (z2 - z1) ** 2)


def exponential_function(*, x: float, base: float = 2.0) -> float:
    return base**x


print(euclidean_distance_3d(1.0, 2.0, 3.0, 4.0, 5.0, 6.0))
print(exponential_function(x=3, base=2))


### 1.3. Типизация и `Callable`

Аннотации говорят читателю и IDE, что входит и что выходит. Функция — такой же
тип, как остальные: `Callable[[int], int]` — «принимает int, возвращает int».
Пример из лекции: функция, принимающая функцию и возвращающая новую, — и это
нормальный, рабочий тип:


In [ ]:
from typing import Callable


def create_number_transformer(
    transform_func: Callable[[int], int],
) -> Callable[[list[int]], dict[str, list[int] | int]]:
    def transformer(numbers: list[int]) -> dict[str, list[int] | int]:
        transformed = [transform_func(n) for n in numbers]
        return {
            "original": numbers,
            "transformed": transformed,
            "total": sum(transformed),
        }

    return transformer


def double(x: int) -> int:
    return x * 2


create_number_transformer(double)([1, 2, 3])


### 1.4. Изменяемый дефолт — ловушка лекции

Дефолтное значение создаётся **один раз**, при объявлении функции (подробнее —
2.3). Изменяемый объект в дефолте живёт между вызовами и накапливает изменения:


In [ ]:
def append_hello(to: list[str] = []) -> None:
    to.append("Hi!")
    print(to)


append_hello()
append_hello()   # дефолт один на все вызовы — «Hi!» задвоился


In [ ]:
def append_hello(to: list[str] | None = None) -> None:
    if to is None:
        to = []
    to.append("Hi!")
    print(to)


append_hello()
append_hello()


Правило лекции: «пустое» значение по умолчанию — `None`, настоящий объект
создаётся в теле функции. Якорь: 2.3 ниже, ДЗ `collect`.

### 1.5. Распаковка аргументов

Вызов `f(*positional, **key_value)` раскладывает последовательность по позициям,
словарь — по именам. Зеркально внутри объявления: `*args` собирает лишние
позиционные в кортеж, `**kwargs` — лишние именованные в словарь:


In [ ]:
def report(x: int, y: int, /, *, option1: int = 0, option2: int = 0) -> None:
    print(x, y, option1, option2)


positional = [4, 8]
key_value = {"option1": 15, "option2": 16}
report(*positional, **key_value)


In [ ]:
def create_mythical_creature(
    species: str,
    name: str,
    /,
    *abilities: str,
    habitat: str = "unknown",
    **traits: object,
) -> dict[str, object]:
    return {
        "species": species,
        "name": name,
        "abilities": list(abilities),
        "habitat": habitat,
        "traits": traits,
    }


dragon = create_mythical_creature(
    "dragon", "Smaug", "fire breath", "flight",
    habitat="Lonely Mountain", wingspan=20,
)
dragon


### 1.6. Области видимости

Четыре пространства имён: **local**, **global**, **non-local**, **builtins**
(non-local — вместе с замыканиями на лекции 11, здесь его нет). Глобальную
функция читает свободно; записать может только через `global`. И главное: в
функцию уходит не копия объекта, а **ссылка** на него:


In [ ]:
c, d = 1, 2
lst1 = [3, 4]


def extend(a: int, b: int, lst: list[int]) -> None:
    lst += [a, b]        # изменили объект — увидит вызывающий


extend(c, d, lst1)
lst1


In [ ]:
e = 0


def set_global() -> None:
    global e
    e = 5


set_global()
e


### 1.7. Класс: тип, класс, объект

**Тип** — контракт на значения и операции, **класс** — его реализация, **объект**
— экземпляр класса. Классы называют в CapWords. Объект наполняется в `__init__`,
`self` — сам объект, `__dict__` — его персональные поля:


In [ ]:
class Robot:
    """Минимальный класс с полем и методом."""

    def __init__(self, name: str) -> None:
        self.name = name          # поле объекта

    def speak(self) -> str:
        return f"Я {self.name}"


r = Robot("Валли")
print(type(r).__name__, r.__class__ is Robot)
print(r.__dict__)
r.speak()


### 1.8. Поле класса и поле объекта

Поле в теле класса — общее, лежит в `Класс.__dict__`; поле через `self` — личное,
в `объект.__dict__`. При чтении `obj.field` поиск идёт сначала в объект, потом в
класс — отсюда конфликт имён из лекции. Метод — обычная функция класса:
`a.die()` это то же самое, что `SampleClass.die(a)`.


In [ ]:
class SampleClass:
    var = ...                    # поле класса — одно на всех

    def __init__(self, var: object) -> None:
        self.var = var           # поле объекта — своё у каждого


a = SampleClass(1)
print(a.var, SampleClass.var)
print("var" in a.__dict__, "var" in SampleClass.__dict__)


### 1.9. Dunder-методы и хешируемость

`__init__`, `__dict__`, `__doc__` — dunder-методы (double underscore): они
подключают объект к механике самого языка. Ключом словаря годится хешируемый
объект — с работающей парой `__hash__` + `__eq__`. У свежего класса оба есть по
умолчанию, поэтому и класс, и его объект — легальные ключи:


In [ ]:
print("__hash__" in dir(SampleClass), "__eq__" in dir(SampleClass))

d = {SampleClass: "класс", a: "объект"}
len(d)


---

## Часть 2. Углубление

### 2.0. Мутация против переприсваивания

Якорь всего семинара. В функцию уходит ссылка на объект вызывающего, и дальше
есть два разных действия. **Изменить объект** (`append`, `+=` у списка, `extend`)
— изменение увидит каждый, у кого есть имя на этот объект. **Переприсвоить имя**
(`values = ...`) — отцепится только локальное имя, вызывающий ничего не заметит:


In [ ]:
def in_place(values: list[int], extra: list[int]) -> None:
    values += extra          # list.__iadd__ — изменение на месте


def rebind(values: list[int], extra: list[int]) -> list[int]:
    values = values + extra  # новый список, привязанный к локальному имени
    return values


first = [1, 2]
in_place(first, [3])
print(first)

second = [1, 2]
print(rebind(second, [3]))
print(second)


Одна строка разницы — противоположные последствия. Раздел 1 семинара; ДЗ `scope`
целиком об этом (`extend_in_place`, `rebind`, `extend_copy`).

### 2.1. Локальность имени решается до запуска

Присваивание делает имя локальным **во всей функции** — не со строки
присваивания, а с самого разбора. Раз в теле есть `total = ...`, то даже `total`
в правой части читает локальную, ещё не созданную переменную:


In [ ]:
total = 10


def increase() -> None:
    total = total + 1  # noqa: F823


try:
    increase()
except UnboundLocalError as e:
    print("UnboundLocalError:", e)


Ошибка не «имени нет» (это `NameError`) — «локальное имя ещё без значения». Это
видно даже без запуска, в байткоде: читающая функция грузит `total` как
глобальную, присваивающая — как локальную:


In [ ]:
import dis


def just_read() -> int:
    return total


for func in just_read, increase:
    ops = [i.opname for i in dis.get_instructions(func) if i.argval == "total"]
    print(f"{func.__name__}: {ops}")


In [ ]:
def increased(value: int) -> int:
    return value + 1


total = increased(total)
total


Лечение — `global total`, но лучше вовсе без глобального состояния: значение
пришло аргументом и ушло результатом. Раздел 1 семинара.

### 2.2. Аннотации ничего не проверяют

При выполнении Python аннотации игнорирует. Это запись о намерении — для
читателя, IDE и `mypy`. Передали строку вместо `int` — строка честно умножится:


In [ ]:
def double(x: int) -> int:
    return x * 2


print(double("ку"))
print(double.__annotations__)


Аннотации — обычный словарь, атрибут функции. Ловит такое только статический
анализатор (`mypy`), не запуск. Раздел 2 семинара; задача семинара `about.py`.

### 2.3. Дефолт вычисляется в момент `def`

Значение по умолчанию — выражение, и вычисляют его **один раз**, когда исполняется
строка `def`. Побочный эффект в дефолте срабатывает без единого вызова:


In [ ]:
def greet(name: str, stamp=print("дефолт создан")) -> None: ...


In [ ]:
greet("Аня")            # тишина: дефолт уже готов
print(greet.__defaults__)


In [ ]:
def collect(value: int, into: list[int] = []) -> list[int]:  # noqa: B006
    into.append(value)
    return into


print(collect(1))
print(collect(2))       # тот же список из дефолта
print(collect.__defaults__[0] is collect(3))


Отсюда паттерн из 1.4: дефолт — `None`, список создаётся в теле при каждом
вызове. ДЗ `collect` требует ровно этого поведения. Раздел 3 семинара покажет ту
же механику на классах.

### 2.4. Функция — объект с атрибутами

У функции-объекта есть `__name__`, `__doc__`, `__annotations__` — и можно вешать
свои. Она хешируема — годится ключом словаря; второе имя на функции ведёт себя
как второе имя на списке:


In [ ]:
def is_even(n: int) -> bool:
    """Проверяет число на чётность."""
    return n % 2 == 0


is_even.purpose = "чек-ин"       # свой атрибут — просто так

alias = is_even
print(alias is is_even, alias(4))
print(is_even.__name__, is_even.__doc__, is_even.purpose)
print({is_even: "функция-ключ"}[alias])


Функцию передают аргументом (в 1.3 — дважды), возвращают, кладут в структуры.
Задача семинара `about.py` — вытащить всё это, не вызывая функцию. Родственно
Prep4, раздел 2.10.

### 2.5. Словарь функций вместо цепочки `if`

Раз функция — значение, ей самое место в словаре. Разбор команды из цепочки
`if/elif` превращается в поиск по ключу, а список поддерживаемых команд
становится данными:


In [ ]:
from collections.abc import Callable


def add(a: int, b: int) -> int:
    return a + b


def subtract(a: int, b: int) -> int:
    return a - b


OPERATIONS: dict[str, Callable[[int, int], int]] = {
    "плюс": add,
    "минус": subtract,
}

print(OPERATIONS["плюс"](2, 3), sorted(OPERATIONS))

OPERATIONS["умножить"] = lambda a, b: a * b   # новая команда — без правки разбора
OPERATIONS["умножить"](2, 3)


Значением может лежать и связанный метод из 2.8 — тогда таблица команд работает
с состоянием конкретного объекта. Раздел 5 семинара; задачи `dispatch.py` и
ДЗ `registry`.

### 2.6. `functools`: кэш и частичное применение

Наивный рекурсивный Фибоначчи считает одно и то же по многу раз.
`functools.lru_cache` запоминает результаты по аргументам — дерево вызовов
схлопывается в линейный проход:


In [ ]:
from functools import lru_cache

count = 0


def fib_counted(n: int) -> int:
    global count
    count += 1
    return n if n < 2 else fib_counted(n - 1) + fib_counted(n - 2)


fib_counted(20)
print("вызовов без кэша:", count)

count = 0


@lru_cache(maxsize=None)
def fib_cached(n: int) -> int:
    global count
    count += 1
    return n if n < 2 else fib_cached(n - 1) + fib_cached(n - 2)


fib_cached(20)
print("вызовов с кэшем:", count)
print("fib(300) с кэшем — сразу:", str(fib_cached(300))[:20] + "…")


In [ ]:
from functools import partial


def power(base: int, exponent: int) -> int:
    return base**exponent


squares = partial(power, exponent=2)   # зафиксировали именованный
of_two = partial(power, 2)             # зафиксировали позиционный

print(squares(5), of_two(10))


Кэш внутри — обычный словарь «аргументы → результат» (лекция 3 ожила).
Частичное применение из второго примера — «функция попроще из функции посложнее»;
в ДЗ `apply` свой `partial` предстоит написать руками.

### 2.7. Поиск атрибута: объект, потом класс

Чтение `obj.field` идёт по цепочке: сначала `obj.__dict__`, затем
`Класс.__dict__`. Поэтому изменяемое поле в теле класса — один список на всех:


In [ ]:
class Basket:
    items: list[str] = []     # создан один раз — при выполнении тела класса

    def add(self, item: str) -> None:
        self.items.append(item)


first, second = Basket(), Basket()
first.add("яблоко")
print(first.items, second.items)
print(first.items is second.items, first.__dict__)


Словарь объекта пуст: своего `items` у него никогда и не было. Число здесь
маскировало бы проблему: `self.total += 1` не меняет число (числа неизменяемы),
а создаёт объекту **своё** поле, затеняющее поле класса:


In [ ]:
class Counter:
    total = 0

    def bump(self) -> None:
        self.total += 1        # self.total = self.total + 1 → поле объекта


left, right = Counter(), Counter()
left.bump()
left.bump()
right.bump()
print(Counter.total, left.total, right.total)
print(left.__dict__, right.__dict__)


Правило: изменяемое состояние создаётся в `__init__`, тогда список у каждого
свой:


In [ ]:
class Basket:
    def __init__(self) -> None:
        self.items: list[str] = []

    def add(self, item: str) -> None:
        self.items.append(item)


first, second = Basket(), Basket()
first.add("яблоко")
print(first.items, second.items, first.items is second.items)


Раздел 3 семинара; ДЗ `registry` (общий `created` + личный `items`), задача
семинара `notes.py`.

### 2.8. `obj.method` — связанный метод

Возьмём метод без вызова. В классе лежит обычная функция; через точку у объекта
достаётся **связанный метод** — обёртка, в которую уже запаян `self`:


In [ ]:
basket = Basket()
add_to_basket = basket.add       # скобок нет — не вызвали

add_to_basket("груша")           # basket не упоминается…
print(basket.items)

print(type(Basket.add).__name__, type(basket.add).__name__)
print(basket.add.__self__ is basket, basket.add.__func__ is Basket.add)


Груша попала именно в его корзину. Связанный метод — обычное значение: его кладут
в таблицу команд из 2.5 и зовут позже. Раздел 4 семинара.

### 2.9. Контракт `__eq__` и `__hash__`

Хешируемый объект обязан выполнять контракт: **равные объекты дают равные хеши**
— иначе словарь клал бы ключ в один слот, а искал бы в другом. Поэтому,
определив свой `__eq__`, вы автоматически теряете хеш — Python не рискует и
снимает его:


In [ ]:
class Point:
    def __init__(self, x: int, y: int) -> None:
        self.x, self.y = x, y

    def __eq__(self, other: object) -> bool:
        return isinstance(other, Point) and (self.x, self.y) == (other.x, other.y)


print(Point(1, 2) == Point(1, 2))
print(Point.__hash__ is None)

try:
    {Point(1, 2): "не бывать"}
except TypeError as e:
    print("TypeError:", e)


In [ ]:
class Point:
    def __init__(self, x: int, y: int) -> None:
        self.x, self.y = x, y

    def __eq__(self, other: object) -> bool:
        return isinstance(other, Point) and (self.x, self.y) == (other.x, other.y)

    def __hash__(self) -> int:
        return hash((self.x, self.y))   # из того, что не меняется


points = {Point(1, 2): "первый", Point(3, 4): "второй"}
points[Point(1, 2)]


Хеш строим из неизменяемого: кортеж координат (мостик к лекции 3 — кортеж
хешируем, пока хешируемо содержимое). ДЗ `vector` — ровно этот контракт.

### 2.10. Мешок трюков

`__init__` обязан вернуть `None` — объект и так уже создан, возвращать ему нечего:


In [ ]:
class Broken:
    def __init__(self) -> None:
        return 42


try:
    Broken()
except TypeError as e:
    print("TypeError:", e)


In [ ]:
def move(distance: int, /, **options: object) -> tuple[int, dict[str, object]]:
    return distance, options


move(5, distance="fast")


In [ ]:
d = {1: 2, 3: 4}
{**d | {1: 5}}


Трюк с `move` работает только из-за `/`: по имени `distance` передать нельзя,
значит имя свободно и уходит в `**options` (ДЗ `signature`). Слияние словарей —
`|` и `{**a, **b}` из лекции.


---

## Часть 3. Викторина «предскажи, что выведет»

Десять вопросов — по recap'у и углублению, функции и классы пополам. Порядок
действий: предскажи вывод, включая исключение, если оно будет → запусти ячейку →
раскрой разбор. 10 из 10 без единого запуска — к семинару готов.


### Вопрос 1

Дефолт и два вызова:


In [ ]:
def collect(value: int, into: list[int] = []) -> list[int]:
    into.append(value)
    return into


print(collect(1))
print(collect(2))


<details>
<summary><b>Разбор</b></summary>

`[1]`, затем `[1, 2]`. Список в дефолте создан один раз — при `def` (2.3); оба
вызова дописывают в него, второй видит результат первого. Фикс — `None` в
дефолте и `if into is None: into = []`.

</details>


### Вопрос 2

Эта ячейка падает. С каким исключением?


In [ ]:
total = 10


def increase() -> None:
    total = total + 1


increase()


<details>
<summary><b>Разбор</b></summary>

`UnboundLocalError: cannot access local variable 'total' where it is not
associated with a value`. Присваивание делает `total` локальным во всей функции
(2.1), поэтому в правой части читается ещё не созданная локальная, а не
глобальная десятка. Не `NameError`: имя в модуле есть, функция про него не знает.

</details>


### Вопрос 3

Аннотации пообещали `int`:


In [ ]:
def double(x: int) -> int:
    return x * 2


double("ку")


<details>
<summary><b>Разбор</b></summary>

`'куку'`. Аннотации при выполнении игнорируются (2.2) — строка честно
умножилась. Поймал бы `mypy`, но не запуск.

</details>


### Вопрос 4

Что вернёт каждый из двух вызовов?


In [ ]:
def linear(x: float, /, slope: float = 1.0) -> float:
    return slope * x


linear(2, slope=10), linear(x=2)


<details>
<summary><b>Разбор</b></summary>

Первый вызов — `20`: аннотации не приводят типы (2.2), `10 * 2` — честный int,
а не обещанный `float`. Второй — `TypeError: linear() got some positional-only
arguments passed as keyword arguments: 'x'`: всё до `/` передаётся только по
позиции (1.1–1.2), `x=` — незаконный способ.

</details>


### Вопрос 5

Тот самый трюк с освобождённым именем:


In [ ]:
def move(distance: int, /, **options: object) -> tuple[int, dict]:
    return distance, options


move(5, distance="fast")


<details>
<summary><b>Разбор</b></summary>

`(5, {'distance': 'fast'})`. Пятёрка ушла в позиционный параметр, а строка — в
`options` под тем же именем: раз `distance` только позиционный, его имя свободно
и не конфликтует с `**kwargs` (2.10). Без `/` такой вызов упал бы с `TypeError:
multiple values`.

</details>


### Вопрос 6

Список в теле класса:


In [ ]:
class Basket:
    items: list[str] = []

    def add(self, item: str) -> None:
        self.items.append(item)


first, second = Basket(), Basket()
first.add("яблоко")

print(second.items, first.items is second.items)


<details>
<summary><b>Разбор</b></summary>

`['яблоко'] True`. Список один, он лежит в классе (2.7); своего `items` у
объектов нет, оба читают класс. Лечится созданием списка в `__init__`.

</details>


### Вопрос 7

Счётчик как поле класса:


In [ ]:
class Counter:
    total = 0

    def bump(self) -> None:
        self.total += 1


left, right = Counter(), Counter()
left.bump()
right.bump()
right.bump()

print(Counter.total, left.total, right.total)


<details>
<summary><b>Разбор</b></summary>

`0 1 2`. Числа неизменяемы: `self.total += 1` создаёт поле объекта и затеняет
поле класса (2.7). Классовое `total` ни разу не тронуто, у каждого объекта —
свой счёт.

</details>


### Вопрос 8

Метод взяли без вызова и позвали позже:


In [ ]:
class Basket:
    def __init__(self) -> None:
        self.items: list[str] = []

    def add(self, item: str) -> None:
        self.items.append(item)


green, blue = Basket(), Basket()
take = green.add
take("груша")

print(green.items, blue.items)


<details>
<summary><b>Разбор</b></summary>

`['груша'] []`. `green.add` — связанный метод с запаянным `self` (2.8): груша
уехала в корзину `green`, до которой в вызове дела нет. `blue` не при делах.

</details>


### Вопрос 9

Конструктор и подозрительный `return`:


In [ ]:
class Robot:
    def __init__(self, name: str) -> None:
        self.name = name
        return name


Robot("Валли")


<details>
<summary><b>Разбор</b></summary>

`TypeError: __init__() should return None, not 'str'` (2.10). Объект уже создан,
`__init__` только наполняет его — возвращать ему нечего. Поле `name` при этом
успело записаться, но объект вам не достался.

</details>


### Вопрос 10

Общий счётчик созданных объектов:


In [ ]:
class Registry:
    created = 0

    def __init__(self, name: str) -> None:
        self.name = name
        Registry.created += 1


a = Registry("первый")
b = Registry("второй")

print(Registry.created, a.created, b.created)


<details>
<summary><b>Разбор</b></summary>

`2 2 2`. Инкремент идёт явно по классу — поле общее (2.7). Через экземпляр оно
тоже читается: своего `created` у объекта нет, поиск уходит в класс. Это ДЗ
`registry`.

</details>


---

Дальше — `Seminar5.ipynb`: те же темы громче и ближе. С собой возьми вопросы,
которые остались после викторины.
